# What you're building

Every AI agent runs the same loop:

```
   1. PERCEIVE  →  2. DECIDE  →  3. ACT  →  (remember)  →  repeat
   (read input)   (pick a tool)  (do it & reply)
```

A real agent uses a language model for the **DECIDE** step. You'll use `if/else`. Everything else is identical to a real agent — you're building the skeleton.

### The plan (each row = something you'll write)

| Part | What it does | Concept |
|---|---|---|
| State / memory | what the agent knows & remembers | `dict` + `list` |
| `perceive()` | clean the input | `strings` |
| `decide_action()` | choose a tool | `strings` + `if/else` |
| tool functions | the actual skills | `functions` |
| the loop | run it forever | `while` loop |

Work top to bottom. Later cells depend on earlier ones.

## Step 1 — Create the agent's memory (its state)

**Your task:** create a `dict` named `agent` that holds the agent's whole state. It must contain:

- `"name"` → a string (pick any name)
- `"history"` → an **empty list** (will store past turns)
- `"notes"` → an **empty dict** (things the user asks it to remember)
- `"facts"` → a **dict** with at least **3** entries, e.g. `"python"` → a short explanation

**Hints**
- A dict can contain a list and other dicts inside it.
- Empty list is `[]`, empty dict is `{}`.

> 🔗 **Real-agent parallel:** real agents carry a "memory" / "context" object — a bundle of state exactly like this.

In [1]:
# TODO: build the agent state dict described above.
agent = {
    "name": "???",      # <- give it a name
    "history": ???,     # <- empty list
    "notes": ???,       # <- empty dict
    "facts": {
        # <- add at least 3 "topic": "explanation" pairs
    }
}

print(agent["name"])
print("Facts I know:", list(agent["facts"].keys()))

SyntaxError: invalid syntax (696580290.py, line 4)

✅ **Self-check:** it should print your agent's name and a list of at least 3 fact topics, with no error.

## Step 2 — `perceive()`

**Your task:** write a function `perceive(text)` that returns the text with leading/trailing spaces removed. (That's the whole job — an easy first win.)

**Hints**
- One string method does this. Think back to the strings lesson: which method trims spaces?
- Return the result; don't print it.

In [2]:
def perceive(text):
    # TODO: return the input text with surrounding spaces removed
    pass

print(perceive("   hello there   "))

None


✅ **Self-check:** should print `hello there` (no extra spaces). If it prints `None`, you forgot to `return`.

## Step 3 — `decide_action()` (the brain)

**Your task:** write a function `decide_action(text)` that looks at the input and **returns an intent word** telling the rest of the program what to do.

Use these rules (check the **first word** unless noted):

| If the input… | return |
|---|---|
| first word is `bye`, `quit`, or `exit` | `"exit"` |
| first word is `hello`, `hi`, or `hey` | `"greet"` |
| first word is `help` | `"help"` |
| first word is `calc` or `calculate` | `"calculate"` |
| first word is `remember` | `"remember"` |
| first word is `recall` | `"recall"` |
| first word is `history` | `"history"` |
| contains the phrase `what is` (anywhere) | `"lookup"` |
| none of the above | `"unknown"` |

**Hints**
- Lowercase the text first so `"Hello"` and `"hello"` both match: `.lower()`.
- Split into words and look at `words[0]` for the "first word" rules.
- `first in ["hello", "hi", "hey"]` checks several options at once.
- For the `what is` rule, use `"what is" in low` (substring check).
- Order matters: check the specific rules before the final `return "unknown"`.

I've started it for you with the setup and **one** branch as an example — you finish the rest.

In [3]:
def decide_action(text):
    low = text.lower()
    words = low.split()

    if len(words) == 0:
        return "unknown"

    first = words[0]

    # --- example branch (already done for you) ---
    if first in ["bye", "quit", "exit"]:
        return "exit"

    # TODO: add branches for greet, help, calculate,
    #       remember, recall, history, and lookup ("what is")

    return "unknown"

# Tests:
print(decide_action("Hello"))          # greet
print(decide_action("calc 5 + 3"))     # calculate
print(decide_action("what is python")) # lookup
print(decide_action("blah"))           # unknown

unknown
unknown
unknown
unknown


✅ **Self-check:** the four prints should be `greet`, `calculate`, `lookup`, `unknown`.

## Step 4 — The tools (one function each)

Each tool is a function that **returns a string** (the agent's reply). Build them one at a time and test each before moving on.

### Tool 4a — `help_text()`
**Your task:** write `help_text()` that returns a multi-line string listing the commands the agent supports (greet, calc, remember, recall, what is, history, bye).

**Hints**
- Return one string. You can put `\n` inside a string to make new lines.
- Triple-quoted strings `"""..."""` can span multiple lines too.

In [4]:
def help_text():
    # TODO: return a string describing the available commands
    pass

print(help_text())

None


✅ **Self-check:** prints your menu of commands.

### Tool 4b — `calculate(text)`  (numbers + strings + if/else)

**Your task:** the user types something like `calc 12 + 4`. Return the numeric result.

**What it must do**
1. Split the text into words.
2. If the first word is `calc`/`calculate`, drop it (keep only the 3 remaining pieces).
3. If you don't end up with exactly 3 pieces, return a helpful message like `"Use the format: calc 12 + 4"`.
4. Read the two numbers and the operator by index, convert numbers with `float(...)`.
5. With `if/elif`, return `a+b`, `a-b`, `a*b`, or `a/b` based on the operator.
6. **Guard:** if dividing and `b == 0`, return `"I can't divide by zero!"`.

**Hints**
- `words[1:]` gives you all words *except* the first (drops `calc`).
- `len(words) != 3` checks the piece count.
- Convert with `float(words[0])`; the operator `words[1]` stays a string.

In [5]:
def calculate(text):
    # TODO: implement the steps above
    pass

print(calculate("calc 12 + 4"))   # expect 16.0
print(calculate("10 / 0"))        # expect: I can't divide by zero!
print(calculate("calc 9 * 3"))    # expect 27.0

None
None
None


✅ **Self-check:** `16.0`, the divide-by-zero message, then `27.0`.

### Tools 4c — `remember(agent, text)` and `recall(agent, text)`

These let the agent learn from the user. `remember city Astana` saves a note; `recall city` looks it up.

**`remember` must do**
- Split the text. Format is `remember <key> <value...>`.
- If there are fewer than 3 words, return a message asking for the format.
- `key` is the 2nd word. The **value** is everything after it joined with spaces (so multi-word values work).
- Store it: `agent["notes"][key] = value`. Return a confirmation string.

**`recall` must do**
- Split the text (`recall <key>`). If fewer than 2 words, ask what to recall.
- Look up the key in `agent["notes"]` using **`.get(key, default)`** so it never crashes on a missing key. Return the value, or a "no note for X" message.

**Hints**
- Join leftover words: `" ".join(words[2:])`.
- Remember the dict lesson: `.get()` is the *safe* lookup that avoids `KeyError`.


In [ ]:
def remember(agent, text):
    # TODO
    pass

def recall(agent, text):
    # TODO (use .get so it can't crash)
    pass

print(remember(agent, "remember city Astana"))
print(recall(agent, "recall city"))      # expect: Astana
print(recall(agent, "recall planet"))    # expect: a "no note" message

✅ **Self-check:** a confirmation, then `Astana`, then your "no note" message.

### Tools 4d — `lookup(agent, text)` and `show_history(agent)`

**`lookup` must do**
- The user types `what is <topic>`. Strip the phrase down to just the topic and look it up in `agent["facts"]` with `.get()`. Return the fact, or an "I don't know" message.
- **Hint:** `text.lower().replace("what is", "").replace("?", "").strip()` turns `"What is Python?"` into `"python"`.

**`show_history` must do**
- If `agent["history"]` is empty, return `"No conversation history yet."`.
- Otherwise loop over the history (a **list of dicts**, each like `{"user": ..., "agent": ...}`) and build a readable transcript string.
- **Hint:** collect lines in a list, then `"\n".join(lines)` at the end.


In [ ]:
def lookup(agent, text):
    # TODO
    pass

def show_history(agent):
    # TODO
    pass

print(lookup(agent, "what is python"))   # expect your python fact

✅ **Self-check:** prints the fact you stored for `"python"` in Step 1.

## Step 5 — `act()` and `remember_turn()`  (the dispatcher)

`act` connects the brain's decision to the right tool.

**`act(agent, intent, text)` must do**
- Based on `intent`, call the matching tool and **return its reply string**:
  - `"greet"` → a greeting that uses `agent["name"]`
  - `"help"` → `help_text()`
  - `"calculate"` → `str(calculate(text))`  (wrap in `str()` so numbers print cleanly)
  - `"remember"` → `remember(agent, text)`
  - `"recall"` → `recall(agent, text)`
  - `"lookup"` → `lookup(agent, text)`
  - `"history"` → `show_history(agent)`
  - anything else → a fallback like `"I'm not sure. Type 'help'."`

**`remember_turn(agent, user_text, agent_text)` must do**
- Append a dict `{"user": user_text, "agent": agent_text}` to `agent["history"]`.

I've done the first branch and the `remember_turn` hint — you finish `act`.


In [ ]:
def act(agent, intent, text):
    # --- example branch (done for you) ---
    if intent == "greet":
        return f"Hi! I'm {agent['name']}. Type 'help' to see what I can do."

    # TODO: add elif branches for help, calculate, remember,
    #       recall, lookup, history, and an else fallback.

def remember_turn(agent, user_text, agent_text):
    # TODO: append a {"user":..., "agent":...} dict to agent["history"]
    pass


✅ **Self-check:** `print(act(agent, "greet", "hello"))` should greet you by the agent's name.

## Step 6 — Wire up the loop (scripted demo)

Now connect everything into the **perceive → decide → act → remember** pipeline. We'll first run it on a fixed list of commands so you can test without typing.

**Your task:** fill in the 4 pipeline lines marked `TODO` inside the loop, using the functions you wrote.

In [6]:
# fresh memory for the demo
agent["history"] = []
agent["notes"] = {}

demo_commands = ["hello", "calc 12 + 4", "remember city Astana",
                 "recall city", "what is python", "history", "bye"]

for command in demo_commands:
    print(f"You: {command}")

    clean  = ???      # TODO: 1. PERCEIVE  -> call perceive(command)
    intent = ???      # TODO: 2. DECIDE    -> call decide_action(clean)

    if intent == "exit":
        print("Me:  Goodbye! 👋")
        break

    response = ???    # TODO: 3. ACT       -> call act(agent, intent, clean)
    # TODO: 4. remember the turn -> call remember_turn(agent, clean, response)

    print("Me: ", response)
    print("-" * 35)

SyntaxError: invalid syntax (1986077345.py, line 11)

✅ **Self-check:** you should see the agent greet, compute `16.0`, save & recall `Astana`, explain python, print the history, then say goodbye.

In [ ]:
## Step 7 — Talk to it live

Same pipeline, but reading **your** typed input with `input()` in a `while` loop.

**Your task:** fill in the same 4 pipeline lines. Type `bye` to stop.

## CELL 21 (Code)

```python
agent["history"] = []
agent["notes"] = {}

print(f"{agent['name']} is online. Type 'help', or 'bye' to quit.\n")

while True:
    user_input = input("You: ")

    clean  = ???      # TODO: perceive
    intent = ???      # TODO: decide

    if intent == "exit":
        print("Me:  Goodbye! 👋")
        break

    response = ???    # TODO: act
    # TODO: remember the turn

    print("Me: ", response)


✅ **Self-check:** you can hold a short "conversation" — greet it, do math, teach it a note and recall it, ask `what is <topic>`, then `bye`.

# 🎉 If it works — you built an agent!

You wrote, from scratch: a memory (dict + list), perception, a rule-based brain, tools, and the agent loop. That's the real architecture of an AI agent. The only thing a "real" one changes is the **DECIDE** step — it asks a language model instead of your `if/else`. Everything you wrote around it stays the same.


## 🚀 Extensions (pick a few)

Each only needs concepts you already have. To add a skill you must touch **two** places: add a rule in `decide_action()` **and** a branch in `act()`.

1. **Add a fact** about yourself, then ask `what is <you>`.
2. **New operator:** add `%` (remainder) to `calculate`.
3. **`notes` command:** list every saved note (loop over `agent["notes"].items()`).
4. **Greeting counter:** add a number to the agent's state that increases each greeting, and mention the count.
5. **`repeat <text>` tool (challenge):** echo whatever the user types after `repeat`.

In [ ]:
# Your extensions here.